# Multi-trainer Kaggle GPU notebook

This notebook runs the uploaded `multi_trainer-main` project on Kaggle using the notebook GPU runtime.

It is designed for the current project layout where:

- raw market data is copied/standardised with `src.kaggle_prepare_raw_data`
- features are generated with `src.prepare_mt5_data`
- side-specific direction datasets are generated with `src.prepare_direction_dataset`
- BUY-only and SELL-only models are trained through `local_run_pipeline_all_symbols.py`
- `src.select_live_trading_models` is run incrementally after each symbol/model pair
- unselected epoch checkpoints and bulky training logs can then be pruned before moving on to the next pair

Kaggle cannot run MetaTrader 5, so upload your existing raw M5 CSV data as a Kaggle Dataset. The notebook will prepare those CSVs into the project format and write live-ready staged models to `/kaggle/working/run_outputs/For Live Trading`.


## Kaggle setup checklist

1. Create or open a Kaggle notebook.
2. In **Settings**, enable **Accelerator → GPU**.
3. Add a Kaggle Dataset containing the project code. This can be either:
   - a zip named `multi_trainer-main.zip`, or
   - an extracted folder containing `src/`, `config/`, and `local_run_pipeline_all_symbols.py`.
4. Add a Kaggle Dataset containing your raw M5 CSV files. Either use one CSV per symbol or one combined CSV with a `symbol`/`ticker`/`pair` column.
5. Run the cells from top to bottom.

The notebook defaults to a small smoke test. Once that succeeds, set `RUN_PROFILE = full_train`. The default training mode in this version is incremental: train one symbol/model pair, run live-model selection, stage the selected epoch, then delete the bulky original epoch/checkpoint/log folders before moving on.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import textwrap
import zipfile

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

print('Python:', sys.version)
print('Kaggle input exists:', KAGGLE_INPUT.exists())
print('Kaggle working:', KAGGLE_WORKING)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle input exists: True
Kaggle working: /kaggle/working


## 1. Configure the run

Leave `RUN_PROFILE = smoke_test` or `RUN_PROFILE = "smoke_test"` for the first run. It trains one symbol, one architecture and both sides for a very small number of epochs.

Change to `RUN_PROFILE = full_train` or `RUN_PROFILE = "full_train"` once the notebook has proven that it can see your raw data and the GPU.


In [2]:
# -------------------------
# Main user settings
# -------------------------
# Profile constants.
# These make both of these valid:
#   RUN_PROFILE = "smoke_test"
#   RUN_PROFILE = smoke_test
# Use the quoted string form if you prefer.
smoke_test = "smoke_test"
single_symbol = "single_symbol"
full_train = "full_train"

RUN_PROFILE = smoke_test  # smoke_test, single_symbol, or full_train
TIMEFRAME = "M5"

# The notebook auto-finds this zip under /kaggle/input.
# Override CODE_ZIP_PATH if your filename is different.
CODE_ZIP_NAME = "multi_trainer-main.zip"
CODE_ZIP_PATH = None  # e.g. "/kaggle/input/my-code-dataset/multi_trainer-main.zip"

# Raw data input.
# Leave RAW_INPUT_DIR=None to auto-detect a /kaggle/input dataset folder containing CSVs.
RAW_INPUT_DIR = None  # e.g. "/kaggle/input/my-forex-m5-csvs"

# Optional: use this if your raw CSVs are inside a zip dataset file.
RAW_DATA_ZIP_PATH = None  # e.g. "/kaggle/input/my-forex-m5-csvs/raw_csvs.zip"

# Optional: set this if you have one combined CSV containing a symbol/ticker/pair column.
COMBINED_CSV_PATH = None  # e.g. "/kaggle/input/my-forex-m5-csvs/all_symbols_M5.csv"

# Your reduced 10-symbol Forex universe from this code branch.
DEFAULT_SYMBOLS = [
    "USDCAD", "EURGBP", "EURUSD", "USDJPY", "USDCHF",
    "GBPUSD", "AUDCHF", "GBPCHF", "AUDNZD", "EURCHF",
]

ALL_MODEL_CONFIGS = [
    "config/direction_settings_residual_mlp.yaml",
    "config/direction_settings_tcn.yaml",
    "config/direction_settings_inception_time.yaml",
    "config/direction_settings_small_transformer.yaml",
    "config/direction_settings_mixture_of_experts.yaml",
    "config/direction_settings_llm_transformer.yaml",
    "config/direction_settings_timesnet.yaml",
    "config/direction_settings_tsmixer.yaml",
]

# Train/replay date split. Adjust to match the dates in your uploaded CSVs.
TRAIN_START = "2023-01-01"
TRAIN_END = "2025-01-01"
REPLAY_START = "2025-01-01"
REPLAY_END = "2026-01-01"

# GPU/session safeguards.
# Kaggle normally gives a single GPU, so keep PARALLEL_JOBS=1 unless you know the model fits with more.
PARALLEL_JOBS = 1
PREPARE_WORKERS = 2
FORCE_REBUILD_DATA = True
DRY_RUN = False


# Storage-saving mode.
# True = prepare data once, then train one symbol/model pair at a time, run select_live_trading_models,
#        stage the best epoch(s), and delete bulky unselected checkpoints/logs before the next pair.
# False = run the original all-at-once pipeline and keep all epoch checkpoints/logs.
INCREMENTAL_SELECT_AND_PRUNE = True
PRUNE_AFTER_SELECTION = True
PACKAGE_LIVE_ONLY = True
PACKAGE_PREPARED_DATA = False

# Deployment-selection thresholds. These match your usual post-training command.
SELECT_MIN_TRADES = 50
SELECT_MIN_NET_PIPS = 0.0
SELECT_MIN_AVERAGE_NET_PIPS = 0.5
SELECT_MAX_DRAWDOWN_PIPS = 150.0
SELECT_MAX_DRAWDOWN_TO_NET_RATIO = 2.5

# Optional row limits. Use these for smoke tests or when testing paths.
RAW_MAX_ROWS_PER_SYMBOL = None
FEATURE_MAX_ROWS = None
DIRECTION_MAX_ROWS = None
TRAIN_MAX_ROWS = None

# Normalise the profile name so "Smoke Test", "smoke-test" and "smoke_test" all work.
RUN_PROFILE = str(RUN_PROFILE).strip().lower().replace(" ", "_").replace("-", "_")

RUN_PROFILES = {
    "smoke_test": {
        "symbols": ["EURUSD"],
        "model_configs": ["config/direction_settings_residual_mlp.yaml"],
        "sides": ["buy", "sell"],
        "epochs": 2,
        "batch_size": 256,
        "raw_max_rows_per_symbol": 120_000,
        "feature_max_rows": 120_000,
        "direction_max_rows": 120_000,
        "train_max_rows": 120_000,
    },
    "single_symbol": {
        "symbols": ["EURUSD"],
        "model_configs": ALL_MODEL_CONFIGS,
        "sides": ["buy", "sell"],
        "epochs": 50,
        "batch_size": 512,
    },
    "full_train": {
        "symbols": DEFAULT_SYMBOLS,
        "model_configs": ALL_MODEL_CONFIGS,
        "sides": ["buy", "sell"],
        "epochs": 50,
        "batch_size": 512,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(
        f"Unknown RUN_PROFILE: {RUN_PROFILE!r}. "
        f"Valid options are: {', '.join(RUN_PROFILES)}"
    )

_profile = RUN_PROFILES[RUN_PROFILE]
TRAIN_SYMBOLS = _profile["symbols"]
MODEL_CONFIGS = _profile["model_configs"]
SIDES = _profile["sides"]
EPOCHS = _profile["epochs"]
BATCH_SIZE = _profile["batch_size"]

RAW_MAX_ROWS_PER_SYMBOL = _profile.get("raw_max_rows_per_symbol", RAW_MAX_ROWS_PER_SYMBOL)
FEATURE_MAX_ROWS = _profile.get("feature_max_rows", FEATURE_MAX_ROWS)
DIRECTION_MAX_ROWS = _profile.get("direction_max_rows", DIRECTION_MAX_ROWS)
TRAIN_MAX_ROWS = _profile.get("train_max_rows", TRAIN_MAX_ROWS)

print('Run profile:', RUN_PROFILE)
print('Symbols:', TRAIN_SYMBOLS)
print('Model configs:', MODEL_CONFIGS)
print('Sides:', SIDES)
print('Epochs:', EPOCHS, 'Batch size:', BATCH_SIZE)
print('Row limits:', {
    'RAW_MAX_ROWS_PER_SYMBOL': RAW_MAX_ROWS_PER_SYMBOL,
    'FEATURE_MAX_ROWS': FEATURE_MAX_ROWS,
    'DIRECTION_MAX_ROWS': DIRECTION_MAX_ROWS,
    'TRAIN_MAX_ROWS': TRAIN_MAX_ROWS,
})

print('Incremental select/prune:', INCREMENTAL_SELECT_AND_PRUNE)
print('Prune after selection:', PRUNE_AFTER_SELECTION)


Run profile: smoke_test
Symbols: ['EURUSD']
Model configs: ['config/direction_settings_residual_mlp.yaml']
Sides: ['buy', 'sell']
Epochs: 2 Batch size: 256
Row limits: {'RAW_MAX_ROWS_PER_SYMBOL': 120000, 'FEATURE_MAX_ROWS': 120000, 'DIRECTION_MAX_ROWS': 120000, 'TRAIN_MAX_ROWS': 120000}
Incremental select/prune: True
Prune after selection: True


## 2. Check GPU and install non-Torch dependencies

Kaggle normally already has a CUDA-enabled PyTorch build. This cell deliberately avoids reinstalling `torch`, because replacing Kaggle's preinstalled CUDA build can break GPU detection.


In [3]:
import importlib.util

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        DEVICE = 'cuda'
    else:
        print('WARNING: No GPU detected. Check Kaggle notebook Settings -> Accelerator -> GPU.')
        DEVICE = 'cpu'
except Exception as exc:
    print('Torch import failed:', repr(exc))
    DEVICE = 'cpu'

def package_missing(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is None

missing = []
for import_name, pip_name in [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('yaml', 'PyYAML'),
    ('sklearn', 'scikit-learn'),
    ('joblib', 'joblib'),
]:
    if package_missing(import_name):
        missing.append(pip_name)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All non-Torch requirements are already available.')

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
All non-Torch requirements are already available.


'1'

## 3. Load the project code

This cell now supports both Kaggle layouts:

- a project zip such as `multi_trainer-main.zip`
- an already extracted project folder such as `/kaggle/input/mt5-dataset/multi_trainer-main/multi_trainer-main`

Leave `CODE_ZIP_PATH = None` unless you deliberately want to force a specific zip or folder path.


In [4]:
def is_project_root(path: Path) -> bool:
    """True when this folder looks like the multi_trainer project root."""
    return (
        path.is_dir()
        and (path / 'src').is_dir()
        and (path / 'config').is_dir()
        and (path / 'local_run_pipeline_all_symbols.py').is_file()
    )


def find_project_dirs_under_input() -> list[Path]:
    """Find extracted project folders under /kaggle/input."""
    hits = []
    for p in KAGGLE_INPUT.rglob('*'):
        try:
            if is_project_root(p):
                hits.append(p)
        except PermissionError:
            pass
    # Prefer the shortest path if duplicates/nested copies exist, but all should work.
    return sorted(set(hits), key=lambda x: (len(x.parts), str(x)))


def find_code_zip_under_input(filename: str) -> Path | None:
    hits = sorted(KAGGLE_INPUT.rglob(filename))
    if hits:
        if len(hits) > 1:
            print('Multiple matching code zips found; using:', hits[0])
            for h in hits:
                print('  ', h)
        return hits[0]
    return None


PROJECT_PARENT = KAGGLE_WORKING / 'project_src'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

# Clean old extraction/copy so reruns are predictable.
for old in PROJECT_PARENT.glob('*'):
    if old.is_dir():
        shutil.rmtree(old)
    else:
        old.unlink()

forced_path = Path(CODE_ZIP_PATH) if CODE_ZIP_PATH else None
code_zip = None
project_source_dir = None

if forced_path:
    if not forced_path.exists():
        raise FileNotFoundError(
            f'CODE_ZIP_PATH does not exist: {forced_path}\n'
            'In your Kaggle screenshot the project is an extracted folder, not a zip. '
            'Set CODE_ZIP_PATH = None and rerun this cell, or set it to the actual folder path.'
        )
    if forced_path.is_file() and forced_path.suffix.lower() == '.zip':
        code_zip = forced_path
    elif forced_path.is_dir():
        # The forced path might be the project root or a parent containing it.
        if is_project_root(forced_path):
            project_source_dir = forced_path
        else:
            nested = sorted(
                [p for p in forced_path.rglob('*') if is_project_root(p)],
                key=lambda x: (len(x.parts), str(x)),
            )
            if not nested:
                raise RuntimeError(f'CODE_ZIP_PATH folder does not contain a project root: {forced_path}')
            project_source_dir = nested[0]
    else:
        raise RuntimeError(f'CODE_ZIP_PATH is not a zip file or folder: {forced_path}')
else:
    # First try the zip name for backwards compatibility.
    code_zip = find_code_zip_under_input(CODE_ZIP_NAME)
    # If no zip exists, support Kaggle's common extracted-folder dataset layout.
    if code_zip is None:
        project_dirs = find_project_dirs_under_input()
        if not project_dirs:
            raise FileNotFoundError(
                'Could not find either:\n'
                f'  1) {CODE_ZIP_NAME!r} under /kaggle/input, or\n'
                '  2) an extracted project folder containing src/, config/, and local_run_pipeline_all_symbols.py.\n\n'
                'Check the Kaggle dataset is attached, or set CODE_ZIP_PATH to the actual zip/folder path.'
            )
        project_source_dir = project_dirs[0]
        if len(project_dirs) > 1:
            print('Multiple extracted project folders found; using:', project_source_dir)
            for p in project_dirs:
                print('  ', p)

if code_zip is not None:
    print('Using project zip:', code_zip)
    with zipfile.ZipFile(code_zip, 'r') as zf:
        zf.extractall(PROJECT_PARENT)
else:
    print('Using extracted project folder:', project_source_dir)
    dst = PROJECT_PARENT / project_source_dir.name
    shutil.copytree(project_source_dir, dst)

project_candidates = sorted(
    [p for p in PROJECT_PARENT.rglob('*') if is_project_root(p)],
    key=lambda x: (len(x.parts), str(x)),
)
if not project_candidates:
    raise RuntimeError(f'Could not find project root with src/ under {PROJECT_PARENT}')

PROJECT_ROOT = project_candidates[0]
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('Project files:', [
    p.name for p in PROJECT_ROOT.iterdir()
    if p.name in ['src', 'config', 'local_run_pipeline_all_symbols.py', 'requirements.txt']
])


Using extracted project folder: /kaggle/input/datasets/chrisrose100/mt5-dataset/multi_trainer-main/multi_trainer-main
Project root: /kaggle/working/project_src/multi_trainer-main
Project files: ['local_run_pipeline_all_symbols.py', 'config', 'requirements.txt', 'src']


## 4. Find or extract the raw CSV data

Supported inputs:

- one CSV per symbol, for example `EURUSD_M5.csv`, `GBPUSD_M5.csv`
- one combined CSV with a symbol-like column such as `symbol`, `ticker`, `pair`
- optionally a zip containing CSVs, if `RAW_DATA_ZIP_PATH` is set


In [5]:
def contains_csvs(path: Path) -> bool:
    return path.exists() and any(p.suffix.lower() == '.csv' for p in path.rglob('*.csv'))

if RAW_DATA_ZIP_PATH:
    raw_zip = Path(RAW_DATA_ZIP_PATH)
    if not raw_zip.exists():
        raise FileNotFoundError(raw_zip)
    raw_extract_dir = KAGGLE_WORKING / 'raw_csv_input'
    if raw_extract_dir.exists():
        shutil.rmtree(raw_extract_dir)
    raw_extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(raw_zip, 'r') as zf:
        zf.extractall(raw_extract_dir)
    RAW_DIR = raw_extract_dir
elif RAW_INPUT_DIR:
    RAW_DIR = Path(RAW_INPUT_DIR)
else:
    # Prefer folders containing CSVs. This works when the raw CSV dataset is separate
    # or when it is the same dataset as the code zip.
    candidates = []
    for d in sorted(KAGGLE_INPUT.iterdir()):
        if d.is_dir() and contains_csvs(d):
            candidates.append(d)
    if not candidates:
        raise FileNotFoundError(
            'Could not auto-detect a raw CSV folder under /kaggle/input. '
            'Set RAW_INPUT_DIR or RAW_DATA_ZIP_PATH in the settings cell.'
        )
    RAW_DIR = candidates[0]
    if len(candidates) > 1:
        print('Multiple raw CSV folders found; using:', RAW_DIR)
        for c in candidates:
            print('  ', c)

print('Raw data folder:', RAW_DIR)
print('CSV examples:')
for p in sorted(RAW_DIR.rglob('*.csv'))[:20]:
    print('  ', p)

if COMBINED_CSV_PATH:
    print('Combined CSV:', COMBINED_CSV_PATH)


Raw data folder: /kaggle/input/datasets
CSV examples:
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/AUDCAD_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/AUDCHF_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/AUDJPY_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/AUDNZD_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/AUDUSD_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/CADCHF_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/CADJPY_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/CHFJPY_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/EURAUD_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/EURCAD_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/EURCHF_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/EURGBP_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-dataset/raw/EURJPY_M5.csv
   /kaggle/input/datasets/chrisrose100/mt5-datase

## 5. Run the Kaggle training pipeline

This version supports two modes:

- `INCREMENTAL_SELECT_AND_PRUNE = True`: prepare the data once, train one symbol/model pair at a time, run `select_live_trading_models`, stage the selected epoch(s), then prune the bulky original epoch/log folders. This is the recommended Kaggle mode.
- `INCREMENTAL_SELECT_AND_PRUNE = False`: run the original all-symbol/all-model pipeline and keep all epoch checkpoints/logs. This can exceed Kaggle's output storage limit on a full run.

Outputs are written under `/kaggle/working/run_outputs`. The live-ready models are staged under `/kaggle/working/run_outputs/For Live Trading`.


In [6]:

OUTPUT_ROOT = KAGGLE_WORKING / 'run_outputs'
LIVE_ROOT = OUTPUT_ROOT / 'For Live Trading'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LIVE_ROOT.mkdir(parents=True, exist_ok=True)

GENERIC_CONFIG = 'config/direction_settings_generic_multisymbol_31_symbols.yaml'

import csv
import gc
import yaml
from collections import OrderedDict, defaultdict


def run_stream(cmd, *, check=True):
    """Run a subprocess and stream output live so Kaggle does not look idle."""
    print('\nCommand:')
    print(' '.join(map(str, cmd)))
    if DRY_RUN:
        print('[DRY_RUN] Not executing command.')
        return 0
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    rc = process.wait()
    print('\nReturn code:', rc)
    if rc != 0 and check:
        raise RuntimeError(f'Command failed with return code {rc}: ' + ' '.join(map(str, cmd)))
    return rc


def add_common_data_args(cmd):
    if COMBINED_CSV_PATH:
        cmd.extend(['--combined-csv', str(COMBINED_CSV_PATH)])
    if RAW_MAX_ROWS_PER_SYMBOL:
        cmd.extend(['--raw-max-rows-per-symbol', str(RAW_MAX_ROWS_PER_SYMBOL)])
    if FEATURE_MAX_ROWS:
        cmd.extend(['--feature-max-rows', str(FEATURE_MAX_ROWS)])
    if DIRECTION_MAX_ROWS:
        cmd.extend(['--direction-max-rows', str(DIRECTION_MAX_ROWS)])
    if TRAIN_MAX_ROWS:
        cmd.extend(['--train-max-rows', str(TRAIN_MAX_ROWS)])


def maybe_add_force_flags(cmd):
    if FORCE_REBUILD_DATA:
        cmd.extend(['--force-raw', '--force-features'])


def config_token(config_path: str) -> str:
    """Mirror local_run_pipeline_all_symbols._config_token for locating model/log folders."""
    path = Path(config_path)
    cfg_path = path if path.is_absolute() else PROJECT_ROOT / path
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}
    paths = cfg.get('paths', {}) or {}
    model_dir = str(paths.get('model_dir', '') or '')
    if model_dir:
        token = Path(model_dir).name
    else:
        token = path.stem.replace('direction_settings_', '')
    return token.strip().replace(' ', '_') or path.stem


def safe_rmtree(path: Path):
    if path.exists():
        print('Pruning:', path)
        shutil.rmtree(path, ignore_errors=True)


def read_json_if_exists(path: Path, default):
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception as exc:
            print(f'[WARN] Could not read {path}: {exc}')
    return default


def read_csv_rows(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv_rows(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    cols = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                cols.append(key)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=cols, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def finalise_incremental_manifest(master_entries, selection_rows, all_epoch_rows):
    """Rebuild final aggregate manifest after repeated selector calls overwrite the selector manifest."""
    # Deduplicate by live model id, preserving first occurrence.
    dedup = OrderedDict()
    for entry in master_entries:
        mid = entry.get('id') or f"{entry.get('symbol')}_{entry.get('model')}_{entry.get('side')}_{entry.get('epoch')}"
        dedup[mid] = entry
    entries = sorted(dedup.values(), key=lambda x: (x.get('symbol', ''), x.get('model', ''), x.get('side', '')))

    universal_config = LIVE_ROOT / Path(GENERIC_CONFIG).name
    universal_config_path = str(universal_config) if universal_config.exists() else None

    manifest = {
        'created_by': 'kaggle incremental train -> select_live_trading_models -> prune wrapper',
        'live_root': str(LIVE_ROOT),
        'universal_config_path': universal_config_path,
        'selection_logic': {
            'min_trades': SELECT_MIN_TRADES,
            'min_net_pips': SELECT_MIN_NET_PIPS,
            'min_average_net_pips': SELECT_MIN_AVERAGE_NET_PIPS,
            'max_drawdown_pips': SELECT_MAX_DRAWDOWN_PIPS,
            'max_drawdown_to_net_ratio': SELECT_MAX_DRAWDOWN_TO_NET_RATIO,
        },
        'models': entries,
    }
    (LIVE_ROOT / 'live_ensemble_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    (LIVE_ROOT / 'live_ensemble_manifest.yaml').write_text(yaml.safe_dump(manifest, sort_keys=False), encoding='utf-8')
    write_csv_rows(LIVE_ROOT / 'live_model_selection_table.csv', selection_rows)
    write_csv_rows(LIVE_ROOT / 'all_epoch_deployment_scores.csv', all_epoch_rows)

    # Rebuild per-symbol configs.
    by_symbol = defaultdict(list)
    for entry in entries:
        by_symbol[entry.get('symbol', 'UNKNOWN')].append(entry)
    cfg_dir = LIVE_ROOT / 'configs'
    cfg_dir.mkdir(parents=True, exist_ok=True)
    for symbol, items in sorted(by_symbol.items()):
        payload = {
            'symbol': symbol,
            'timeframe': items[0].get('timeframe', TIMEFRAME) if items else TIMEFRAME,
            'live_root': str(LIVE_ROOT),
            'universal_config_path': universal_config_path,
            'models': items,
        }
        (cfg_dir / f'{symbol}_live_ensemble.yaml').write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')

    print(f'Final live manifest entries: {len(entries)}')
    print('Final live manifest:', LIVE_ROOT / 'live_ensemble_manifest.json')
    print('Final live selection table:', LIVE_ROOT / 'live_model_selection_table.csv')
    return manifest


if not INCREMENTAL_SELECT_AND_PRUNE:
    # Original all-at-once mode. This keeps all epochs/logs and can exceed Kaggle storage on a full run.
    cmd = [
        sys.executable, 'local_run_pipeline_all_symbols.py',
        '--raw-input-dir', str(RAW_DIR),
        '--symbols', *TRAIN_SYMBOLS,
        '--timeframe', TIMEFRAME,
        '--configs', *MODEL_CONFIGS,
        '--sides', *SIDES,
        '--mode', 'train-side-all',
        '--train-start', TRAIN_START,
        '--train-end', TRAIN_END,
        '--replay-start', REPLAY_START,
        '--replay-end', REPLAY_END,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--device', DEVICE,
        '--parallel-jobs', str(PARALLEL_JOBS),
        '--prepare-workers', str(PREPARE_WORKERS),
        '--output-root', str(OUTPUT_ROOT),
    ]
    add_common_data_args(cmd)
    maybe_add_force_flags(cmd)
    if DRY_RUN:
        cmd.append('--dry-run')
    run_stream(cmd)
else:
    # Recommended Kaggle mode: prepare once, then train/select/prune one symbol/model pair at a time.
    prepare_cmd = [
        sys.executable, 'local_run_pipeline_all_symbols.py',
        '--raw-input-dir', str(RAW_DIR),
        '--symbols', *TRAIN_SYMBOLS,
        '--timeframe', TIMEFRAME,
        '--configs', *MODEL_CONFIGS,
        '--mode', 'prepare-only',
        '--train-start', TRAIN_START,
        '--train-end', TRAIN_END,
        '--replay-start', REPLAY_START,
        '--replay-end', REPLAY_END,
        '--prepare-workers', str(PREPARE_WORKERS),
        '--output-root', str(OUTPUT_ROOT),
    ]
    add_common_data_args(prepare_cmd)
    maybe_add_force_flags(prepare_cmd)
    if DRY_RUN:
        prepare_cmd.append('--dry-run')
    run_stream(prepare_cmd)

    master_manifest_entries = []
    aggregate_selection_rows = []
    aggregate_all_epoch_rows = []
    selection_runs_dir = LIVE_ROOT / 'selection_runs'
    selection_runs_dir.mkdir(parents=True, exist_ok=True)

    total_pairs = len(MODEL_CONFIGS) * len(TRAIN_SYMBOLS)
    pair_index = 0
    for cfg in MODEL_CONFIGS:
        token = config_token(cfg)
        for symbol in TRAIN_SYMBOLS:
            pair_index += 1
            print('\n' + '=' * 100)
            print(f'[{pair_index}/{total_pairs}] Training then selecting: symbol={symbol} model={token} sides={SIDES}')
            print('=' * 100)

            train_cmd = [
                sys.executable, 'local_run_pipeline_all_symbols.py',
                '--raw-input-dir', str(RAW_DIR),
                '--symbols', symbol,
                '--timeframe', TIMEFRAME,
                '--configs', cfg,
                '--sides', *SIDES,
                '--mode', 'train-side-all',
                '--train-start', TRAIN_START,
                '--train-end', TRAIN_END,
                '--replay-start', REPLAY_START,
                '--replay-end', REPLAY_END,
                '--epochs', str(EPOCHS),
                '--batch-size', str(BATCH_SIZE),
                '--device', DEVICE,
                '--parallel-jobs', str(PARALLEL_JOBS),
                '--prepare-workers', str(PREPARE_WORKERS),
                '--skip-raw-copy',
                '--skip-feature-prep',
                '--skip-direction-prep',
                '--output-root', str(OUTPUT_ROOT),
            ]
            if TRAIN_MAX_ROWS:
                train_cmd.extend(['--train-max-rows', str(TRAIN_MAX_ROWS)])
            if DRY_RUN:
                train_cmd.append('--dry-run')
            run_stream(train_cmd)

            pair_logs_root = OUTPUT_ROOT / 'logs' / token / symbol
            if not pair_logs_root.exists():
                print('[WARN] Expected pair log root missing; falling back to full logs root:', pair_logs_root)
                pair_logs_root = OUTPUT_ROOT / 'logs'

            select_cmd = [
                sys.executable, '-m', 'src.select_live_trading_models',
                '--logs-root', str(pair_logs_root),
                '--project-root', str(OUTPUT_ROOT),
                '--output-root', str(LIVE_ROOT),
                '--generic-config', GENERIC_CONFIG,
                '--symbols', symbol,
                '--models', token,
                '--sides', *SIDES,
                '--min-trades', str(SELECT_MIN_TRADES),
                '--min-net-pips', str(SELECT_MIN_NET_PIPS),
                '--min-average-net-pips', str(SELECT_MIN_AVERAGE_NET_PIPS),
                '--max-drawdown-pips', str(SELECT_MAX_DRAWDOWN_PIPS),
                '--max-drawdown-to-net-ratio', str(SELECT_MAX_DRAWDOWN_TO_NET_RATIO),
            ]
            rc = run_stream(select_cmd, check=False)
            if rc != 0:
                print(f'[WARN] Selection command failed for {symbol} {token}. Continuing so other models can train.')

            # The selector overwrites its manifest each time. Capture this run, then rebuild the aggregate at the end.
            run_manifest = read_json_if_exists(LIVE_ROOT / 'live_ensemble_manifest.json', {'models': []})
            new_entries = run_manifest.get('models', []) if isinstance(run_manifest, dict) else []
            print(f'Selected live-ready models this pair: {len(new_entries)}')
            master_manifest_entries.extend(new_entries)

            pair_tag = f'{symbol}_{token}'
            for side in SIDES:
                pair_tag += f'_{side}'
            selection_table = LIVE_ROOT / 'live_model_selection_table.csv'
            all_epoch_scores = LIVE_ROOT / 'all_epoch_deployment_scores.csv'
            if selection_table.exists():
                rows = read_csv_rows(selection_table)
                aggregate_selection_rows.extend(rows)
                shutil.copy2(selection_table, selection_runs_dir / f'{pair_tag}_live_model_selection_table.csv')
            if all_epoch_scores.exists():
                rows = read_csv_rows(all_epoch_scores)
                aggregate_all_epoch_rows.extend(rows)
                shutil.copy2(all_epoch_scores, selection_runs_dir / f'{pair_tag}_all_epoch_deployment_scores.csv')

            # Rebuild the aggregate files after every pair so an interrupted Kaggle run still has a useful partial manifest.
            finalise_incremental_manifest(master_manifest_entries, aggregate_selection_rows, aggregate_all_epoch_rows)

            if PRUNE_AFTER_SELECTION:
                # Remove bulky original training outputs. Staged live-ready files under LIVE_ROOT are kept.
                safe_rmtree(OUTPUT_ROOT / 'models' / token / symbol)
                safe_rmtree(OUTPUT_ROOT / 'logs' / token / symbol)
                # Generated per-side training configs are small but not needed after selection.
                generated_root = OUTPUT_ROOT / 'config' / 'generated_local'
                for side in SIDES:
                    generated_cfg = generated_root / f'{Path(cfg).stem}_{symbol}_{side}.yaml'
                    if generated_cfg.exists():
                        print('Removing generated training config:', generated_cfg)
                        generated_cfg.unlink()
                gc.collect()
                try:
                    import torch
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception:
                    pass

    final_manifest = finalise_incremental_manifest(master_manifest_entries, aggregate_selection_rows, aggregate_all_epoch_rows)
    print('\nIncremental training/select/prune complete.')



Command:
/usr/bin/python3 local_run_pipeline_all_symbols.py --raw-input-dir /kaggle/input/datasets --symbols EURUSD --timeframe M5 --configs config/direction_settings_residual_mlp.yaml --mode prepare-only --train-start 2023-01-01 --train-end 2025-01-01 --replay-start 2025-01-01 --replay-end 2026-01-01 --prepare-workers 2 --output-root /kaggle/working/run_outputs --raw-max-rows-per-symbol 120000 --feature-max-rows 120000 --direction-max-rows 120000 --train-max-rows 120000 --force-raw --force-features
Pipeline selection: 1 config(s) x 1 symbol(s) x 2 side task(s)
Symbols: EURUSD
Configs: config/direction_settings_residual_mlp.yaml
Sides: buy, sell

$ /usr/bin/python3 -m src.kaggle_prepare_raw_data --input-dir /kaggle/input/datasets --output-dir data/raw --timeframe M5 --symbols EURUSD --max-rows-per-symbol 120000 --force
Wrote 120,000 rows from EURUSD_M5.csv: data/raw/EURUSD_M5.csv
Prepared 1 raw file(s) in /kaggle/working/project_src/multi_trainer-main/data/raw

$ /usr/bin/python3 -m s

## 6. Inspect the training summary

The pipeline writes a JSON summary containing every model/symbol/side task and stdout/stderr tails for any failures.


In [7]:

summary_path = OUTPUT_ROOT / 'logs' / 'local_side_setup_training_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Latest per-pair summary:', summary_path)
    print('Tasks:', len(summary.get('tasks', [])))
    print('Failures:', len(summary.get('failures', [])))
    if summary.get('failures'):
        print('\nFailure preview:')
        for failure in summary['failures'][:5]:
            print(json.dumps({
                'symbol': failure.get('symbol'),
                'side': failure.get('side'),
                'config': failure.get('config'),
                'returncode': failure.get('returncode'),
                'stderr_tail': failure.get('stderr_tail', '')[-1000:],
            }, indent=2))
else:
    print('No latest summary found yet:', summary_path)

manifest_path = LIVE_ROOT / 'live_ensemble_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print('\nLive manifest:', manifest_path)
    print('Live-ready model entries:', len(manifest.get('models', [])))
    by_symbol = {}
    for m in manifest.get('models', []):
        by_symbol.setdefault(m.get('symbol', 'UNKNOWN'), 0)
        by_symbol[m.get('symbol', 'UNKNOWN')] += 1
    print('Models per symbol:', by_symbol)
else:
    print('\nNo live manifest found yet:', manifest_path)

print('\nLive output folders:')
for p in sorted(LIVE_ROOT.glob('*'))[:50]:
    print(' ', p)

print('\nRemaining bulky model folders after pruning:')
models_root = OUTPUT_ROOT / 'models'
if models_root.exists():
    for p in sorted(models_root.glob('*'))[:20]:
        print(' ', p)
else:
    print('  none')


Latest per-pair summary: /kaggle/working/run_outputs/logs/local_side_setup_training_summary.json
Tasks: 2
Failures: 0

Live manifest: /kaggle/working/run_outputs/For Live Trading/live_ensemble_manifest.json
Live-ready model entries: 0
Models per symbol: {}

Live output folders:
  /kaggle/working/run_outputs/For Live Trading/all_epoch_deployment_scores.csv
  /kaggle/working/run_outputs/For Live Trading/configs
  /kaggle/working/run_outputs/For Live Trading/direction_settings_generic_multisymbol_31_symbols.yaml
  /kaggle/working/run_outputs/For Live Trading/generated_spread_risk.yaml
  /kaggle/working/run_outputs/For Live Trading/live_ensemble_manifest.json
  /kaggle/working/run_outputs/For Live Trading/live_ensemble_manifest.yaml
  /kaggle/working/run_outputs/For Live Trading/live_model_selection_table.csv
  /kaggle/working/run_outputs/For Live Trading/selection_runs

Remaining bulky model folders after pruning:
  /kaggle/working/run_outputs/models/residual_mlp


## 7. Package outputs for download

This creates `/kaggle/working/multi_trainer_kaggle_outputs.zip`. With `PACKAGE_LIVE_ONLY=True`, the zip mainly contains the staged `For Live Trading` folder and compact selection summaries, rather than every epoch checkpoint.


In [8]:

package_root = KAGGLE_WORKING / 'package_outputs'
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)

# Always include the live-ready staged folder if present.
if LIVE_ROOT.exists():
    shutil.copytree(LIVE_ROOT, package_root / 'For Live Trading')

if not PACKAGE_LIVE_ONLY:
    for name in ['models', 'logs', 'config']:
        src = OUTPUT_ROOT / name
        if src.exists():
            shutil.copytree(src, package_root / name)
else:
    # Include generated config/selection metadata if present, but avoid bulky model/log trees.
    src = OUTPUT_ROOT / 'config'
    if src.exists():
        shutil.copytree(src, package_root / 'config')

# Prepared data can be very large. Leave PACKAGE_PREPARED_DATA=False for most Kaggle full runs.
if PACKAGE_PREPARED_DATA:
    for rel in ['data/raw', 'data/processed_m5', 'data/direction']:
        src = PROJECT_ROOT / rel
        if src.exists():
            dst = package_root / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src, dst)

zip_base = KAGGLE_WORKING / 'multi_trainer_kaggle_outputs'
zip_path = shutil.make_archive(str(zip_base), 'zip', package_root)
print('Created:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / 1024 / 1024)


Created: /kaggle/working/multi_trainer_kaggle_outputs.zip
Size MB: 0.007594108581542969


## Optional: prepare data only

Use this command instead of the training cell if you only want Kaggle to standardise raw CSVs, prepare features and generate direction-label CSVs. This is useful before a long full training run.


In [9]:
# Uncomment and run manually if you want prepare-only mode.
# prepare_cmd = [
#     sys.executable, 'local_run_pipeline_all_symbols.py',
#     '--raw-input-dir', str(RAW_DIR),
#     '--symbols', *TRAIN_SYMBOLS,
#     '--timeframe', TIMEFRAME,
#     '--configs', *MODEL_CONFIGS,
#     '--mode', 'prepare-only',
#     '--train-start', TRAIN_START,
#     '--replay-end', REPLAY_END,
#     '--prepare-workers', str(PREPARE_WORKERS),
#     '--output-root', str(OUTPUT_ROOT),
# ]
# if COMBINED_CSV_PATH:
#     prepare_cmd.extend(['--combined-csv', str(COMBINED_CSV_PATH)])
# print(' '.join(map(str, prepare_cmd)))
# subprocess.check_call(prepare_cmd)
